# Spoken Grammar Scoring Engine v2 (score 0-5)

**Goal:** predict a continuous grammar score (0-5) from a 45-60 s speech recording.
**Data:** 769 labelled training files, 216 files to predict. **Metrics:** RMSE and Pearson correlation.

## Report

### Approach
Grammar shows up in *what* is said (the words and sentence structure) and *how* it is said (fluency, pronunciation). Each recording is described with four feature groups, several regressors are trained on each group, and their predictions are blended by stacking.

### Preprocessing
- Audio loaded as mono, resampled to **16 kHz**, trimmed to **60 s**, split into 30 s chunks for Whisper.
- **Train and test audio are looked up inside their own folders.** Some filenames exist in both folders, so a name-only lookup silently reads the wrong recording.
- Whisper-small generates an English transcript. Numeric features are standardised before Ridge/SVR.

### Pipeline architecture
`audio -> Whisper-small -> (1) audio embedding, (2) transcript`
`audio -> WavLM-large -> (3) acoustic embedding (middle layers, mean + std)`
`transcript -> (4) RoBERTa embedding, GPT-2 loss and grammar/fluency statistics`
`each feature group -> Ridge / SVR / Gradient Boosting -> 5-fold out-of-fold predictions`
`+ hurdle model: P(score = 0) classifier x regressor trained on non-zero scores`
`-> non-negative linear stacking -> clip to [0, 5]`

### Why the hurdle model
About 5% of the training files are labelled 0 (outside the 1-5 rubric, likely silent or unusable audio). A plain regressor predicts values between 0 and 2 for them, and each such miss costs far more squared error than a typical file. The hurdle model predicts `(1 - P(zero)) x regression`, which is the MSE-optimal prediction when P(zero) is well calibrated.

### Evaluation
5-fold stratified cross-validation (each file is scored by models that never saw it). Reported: **RMSE**, **Pearson**, and the compulsory **training RMSE**. The notebook compares three stacking sets (original models, + WavLM and tuned models, + hurdle models) by CV RMSE and uses the best one.

## 0. Imports and setup

In [ ]:
# ===== All imports =====
import os, re, glob, pickle, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import librosa
from tqdm.auto import tqdm
from scipy.stats import pearsonr
from IPython.display import display, Markdown

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import RidgeCV, Ridge, LogisticRegression
from sklearn.svm import SVR
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_squared_error, roc_auc_score

from transformers import (WhisperProcessor, WhisperForConditionalGeneration,
                          AutoTokenizer, AutoModel, AutoModelForCausalLM, AutoFeatureExtractor)

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

## 1. Load data
Train and test audio are matched **inside their own folders** (some filenames exist in both).

In [ ]:
# ---- Settings ----
DATA_DIR = "/kaggle/input"
SR = 16000
MAX_SEC = 60
WHISPER = "openai/whisper-small"
WAVLM = "microsoft/wavlm-large"
CACHE = "features_cache.pkl"      # Whisper results (resume if the session stops)
WAVLM_CACHE = "wavlm_feats.npy"   # WavLM results

AUDIO_EXT = (".wav", ".mp3", ".flac", ".m4a", ".ogg")
def stem(x):
    return os.path.splitext(os.path.basename(str(x)))[0]

# ---- Find audio files, separated by the folder they live in ----
train_map, test_map = {}, {}
for p in glob.glob(os.path.join(DATA_DIR, "**", "*"), recursive=True):
    if p.lower().endswith(AUDIO_EXT):
        parts = p.replace("\\", "/").lower().split("/")
        if "train" in parts:
            train_map[stem(p)] = p
        elif "test" in parts:
            test_map[stem(p)] = p
print("train audio:", len(train_map), "| test audio:", len(test_map))
assert len(train_map) > 0 and len(test_map) > 0, "Audio not found: check DATA_DIR and that the dataset is added as Input"

# ---- CSVs ----
csvs = sorted(glob.glob(os.path.join(DATA_DIR, "**", "*.csv"), recursive=True))
train_csv = next(c for c in csvs if "train" in os.path.basename(c).lower())
test_csv = next(c for c in csvs if "test" in os.path.basename(c).lower())
print("train csv:", train_csv, "\ntest csv :", test_csv)
train_df = pd.read_csv(train_csv)
test_df = pd.read_csv(test_csv)

FILE_COL = train_df.columns[0]
LABEL_COL = [c for c in train_df.select_dtypes("number").columns if c != FILE_COL][-1]
print("FILE_COL:", FILE_COL, "| LABEL_COL:", LABEL_COL)

train_df["path"] = train_df[FILE_COL].astype(str).map(lambda x: train_map[stem(x)])
test_df["path"] = test_df[FILE_COL].astype(str).map(lambda x: test_map[stem(x)])
y = train_df[LABEL_COL].values.astype(float)

all_df = pd.concat([train_df, test_df], ignore_index=True)
is_train = np.arange(len(all_df)) < len(train_df)
print("train rows:", len(train_df), "| test rows:", len(test_df))
print("paths shared by train and test rows (must be 0):", len(set(train_df.path) & set(test_df.path)))
display(train_df.head(3))

## 2. Label distribution

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(y, bins=20, kde=True, ax=ax[0]); ax[0].set_title("Training score distribution")
sns.countplot(x=np.round(y * 2) / 2, ax=ax[1]); ax[1].set_title("Counts per score")
plt.tight_layout(); plt.show()
print("Baseline RMSE (always predict the mean):", round(np.sqrt(np.mean((y - y.mean()) ** 2)), 4))
print("Files labelled 0:", int((y == 0).sum()), "of", len(y))

## 3. Whisper: transcript + audio embedding
Slowest step (~15-20 min on a T4 GPU). Cached to disk.

In [ ]:
processor = WhisperProcessor.from_pretrained(WHISPER)
dtype = torch.float16 if device == "cuda" else torch.float32
whisper = WhisperForConditionalGeneration.from_pretrained(WHISPER).to(device).to(dtype).eval()
mid_layer = whisper.config.encoder_layers * 2 // 3

@torch.no_grad()
def whisper_features(path):
    audio, _ = librosa.load(path, sr=SR, mono=True, duration=MAX_SEC)
    if len(audio) < SR:
        audio = np.pad(audio, (0, SR - len(audio)))
    chunks = [audio[i:i + 30 * SR] for i in range(0, len(audio), 30 * SR)]
    chunks = [c for c in chunks if len(c) > SR] or [audio]
    inputs = processor.feature_extractor(chunks, sampling_rate=SR, return_tensors="pt").input_features.to(device, dtype)

    hidden = whisper.model.encoder(inputs, output_hidden_states=True).hidden_states
    embs = []
    for k, c in enumerate(chunks):
        n = max(1, min(1500, int(len(c) / SR * 50)))
        embs.append(torch.cat([hidden[mid_layer][k, :n].mean(0), hidden[-1][k, :n].mean(0)]).float().cpu().numpy())
    emb = np.mean(embs, axis=0)

    ids = whisper.generate(inputs, language="en", task="transcribe", max_new_tokens=300)
    text = " ".join(t.strip() for t in processor.batch_decode(ids, skip_special_tokens=True))
    return emb, text, len(audio) / SR

store = pickle.load(open(CACHE, "rb")) if os.path.exists(CACHE) else {}
for i, p in enumerate(tqdm(all_df["path"])):
    if p not in store:
        store[p] = whisper_features(p)
    if i % 50 == 0:
        pickle.dump(store, open(CACHE, "wb"))
pickle.dump(store, open(CACHE, "wb"))

X_audio = np.stack([store[p][0] for p in all_df["path"]])
all_df["transcript"] = [store[p][1] for p in all_df["path"]]
all_df["duration"] = [store[p][2] for p in all_df["path"]]
print("audio embedding shape:", X_audio.shape)
display(all_df[["transcript"]].head(3))

del whisper
torch.cuda.empty_cache()

## 4. Text features: GPT-2 loss, grammar/fluency statistics, RoBERTa embedding

In [ ]:
gpt_tok = AutoTokenizer.from_pretrained("gpt2")
gpt = AutoModelForCausalLM.from_pretrained("gpt2").to(device).eval()

@torch.no_grad()
def gpt_loss(text):
    ids = gpt_tok(text, return_tensors="pt", truncation=True, max_length=512).input_ids.to(device)
    if ids.shape[1] < 2:
        return np.nan
    return gpt(ids, labels=ids).loss.item()

FILLERS = {"um", "uh", "er", "ah", "hmm", "erm"}

def text_features(text, duration):
    words = re.findall(r"[A-Za-z']+", text.lower())
    n = max(len(words), 1)
    sentences = [s for s in re.split(r"[.!?]+", text) if s.strip()]
    lens = [len(s.split()) for s in sentences] or [0]
    sent_loss = [gpt_loss(s) for s in sentences[:25] if len(s.split()) >= 3]
    sent_loss = [v for v in sent_loss if not np.isnan(v)] or [np.nan]
    return {
        "gpt_loss": gpt_loss(text),
        "gpt_sent_mean": np.nanmean(sent_loss),
        "gpt_sent_max": np.nanmax(sent_loss),
        "n_words": len(words),
        "words_per_min": len(words) / max(duration, 1) * 60,
        "type_token_ratio": len(set(words)) / n,
        "n_sentences": len(sentences),
        "sent_len_mean": np.mean(lens),
        "sent_len_max": np.max(lens),
        "filler_rate": sum(w in FILLERS for w in words) / n,
        "repeat_rate": sum(words[i] == words[i - 1] for i in range(1, len(words))) / n,
        "comma_rate": text.count(",") / n,
    }

hand_df = pd.DataFrame([text_features(t, d) for t, d in tqdm(zip(all_df["transcript"], all_df["duration"]), total=len(all_df))])
hand_df = hand_df.fillna(hand_df.median())
X_hand = hand_df.values
del gpt
torch.cuda.empty_cache()

rob_tok = AutoTokenizer.from_pretrained("roberta-base")
roberta = AutoModel.from_pretrained("roberta-base").to(device).eval()

@torch.no_grad()
def roberta_embedding(text):
    batch = rob_tok(text if text else ".", return_tensors="pt", truncation=True, max_length=512).to(device)
    h = roberta(**batch).last_hidden_state[0]
    return torch.cat([h.mean(0), h[0]]).cpu().numpy()

X_text = np.stack([roberta_embedding(t) for t in tqdm(all_df["transcript"])])
del roberta
torch.cuda.empty_cache()
print("shapes:", X_audio.shape, X_text.shape, X_hand.shape)

## 5. WavLM-large acoustic embedding (extra audio view)
A second speech model gives a different view of pronunciation and fluency. ~5-10 min on a T4 GPU, cached to disk.

In [ ]:
if os.path.exists(WAVLM_CACHE):
    X_wavlm = np.load(WAVLM_CACHE)
else:
    wl_fe = AutoFeatureExtractor.from_pretrained(WAVLM)
    wl = AutoModel.from_pretrained(WAVLM).to(device).eval()
    n_l = wl.config.num_hidden_layers
    layers = [n_l // 3, n_l // 2, (2 * n_l) // 3]          # middle layers carry the most linguistic information
    feats = []
    for p in tqdm(all_df["path"]):
        audio, _ = librosa.load(p, sr=SR, mono=True, duration=MAX_SEC)
        if len(audio) < SR:
            audio = np.pad(audio, (0, SR - len(audio)))
        inp = wl_fe(audio, sampling_rate=SR, return_tensors="pt").input_values.to(device)
        with torch.no_grad(), torch.autocast(device_type="cuda", dtype=torch.float16, enabled=(device == "cuda")):
            hs = wl(inp, output_hidden_states=True).hidden_states
        parts = [hs[l][0].float().mean(0) for l in layers] + [hs[layers[1]][0].float().std(0)]
        feats.append(torch.cat(parts).cpu().numpy())
    X_wavlm = np.stack(feats)
    np.save(WAVLM_CACHE, X_wavlm)
    del wl
    torch.cuda.empty_cache()
assert X_wavlm.shape[0] == len(all_df)
print("WavLM embedding shape:", X_wavlm.shape)

## 6. Train models with 5-fold cross-validation
Every model gets out-of-fold predictions (for honest scoring and stacking), a refit on all training data, and test predictions.

In [ ]:
def rmse(a, b):
    return float(np.sqrt(mean_squared_error(a, b)))

y_bins = pd.qcut(y, 5, labels=False, duplicates="drop")        # keeps the score balance in each fold
folds = list(StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED).split(np.zeros(len(y)), y_bins))

oof, train_preds, test_preds, rows = {}, {}, {}, []

def add_model(name, X, make):
    Xtr, Xte = X[is_train], X[~is_train]
    o = np.zeros(len(y))
    for tr_idx, va_idx in folds:
        o[va_idx] = make().fit(Xtr[tr_idx], y[tr_idx]).predict(Xtr[va_idx])
    full = make().fit(Xtr, y)                                   # refit on all training data
    oof[name] = np.clip(o, 0, 5)
    train_preds[name] = np.clip(full.predict(Xtr), 0, 5)
    test_preds[name] = np.clip(full.predict(Xte), 0, 5)
    rows.append((name, rmse(y, oof[name]), pearsonr(y, oof[name])[0]))
    print(f"{name:16s} CV RMSE = {rows[-1][1]:.4f}   Pearson = {rows[-1][2]:.4f}")

ridge = lambda: make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(0, 5, 11)))
svr = lambda C: (lambda: make_pipeline(StandardScaler(), SVR(C=C, epsilon=0.1)))
gboost = lambda: HistGradientBoostingRegressor(max_depth=3, learning_rate=0.05, max_iter=300, random_state=SEED)

# ---- original models ----
add_model("audio_ridge", X_audio, ridge)
add_model("audio_svr", X_audio, svr(1.5))
add_model("text_ridge", X_text, ridge)
add_model("text_svr", X_text, svr(1.5))
add_model("stats_ridge", X_hand, ridge)
add_model("stats_svr", X_hand, svr(1.5))
add_model("stats_gboost", X_hand, gboost)
BASE = list(oof)

# ---- new: WavLM, tuned SVR strength, combined features ----
X_combo = np.hstack([X_audio, X_wavlm, X_text])
add_model("wavlm_ridge", X_wavlm, ridge)
for C in [1, 3, 10]:
    add_model(f"wavlm_svr_C{C}", X_wavlm, svr(C))
for C in [3, 10]:
    add_model(f"audio_svr_C{C}", X_audio, svr(C))
for C in [3, 10]:
    add_model(f"combo_svr_C{C}", X_combo, svr(C))
NEW = [n for n in oof if n not in BASE]

## 7. Hurdle model for the score-0 recordings
Stage 1 predicts P(score = 0). Stage 2 is a regressor trained only on non-zero scores. Prediction = (1 - P(zero)) x stage 2.

In [ ]:
zero = (y == 0).astype(int)
print("zero files:", zero.sum(), "of", len(y))

def hurdle(name, Xz, Xr, C_clf=0.02, C_reg=3):
    Xz_tr, Xz_te = Xz[is_train], Xz[~is_train]
    Xr_tr, Xr_te = Xr[is_train], Xr[~is_train]

    def fit_predict(tr, Xz_b, Xr_b):
        clf = make_pipeline(StandardScaler(), LogisticRegression(C=C_clf, max_iter=3000)).fit(Xz_tr[tr], zero[tr])
        keep = tr[zero[tr] == 0]                                  # stage 2 sees only non-zero scores
        reg = make_pipeline(StandardScaler(), SVR(C=C_reg, epsilon=0.1)).fit(Xr_tr[keep], y[keep])
        p = clf.predict_proba(Xz_b)[:, 1]
        return (1 - p) * np.clip(reg.predict(Xr_b), 0, 5), p

    o, p_oof = np.zeros(len(y)), np.zeros(len(y))
    for tr_idx, va_idx in folds:
        o[va_idx], p_oof[va_idx] = fit_predict(tr_idx, Xz_tr[va_idx], Xr_tr[va_idx])
    all_tr = np.arange(len(y))
    tr_p, _ = fit_predict(all_tr, Xz_tr, Xr_tr)
    te_p, _ = fit_predict(all_tr, Xz_te, Xr_te)
    oof[name], train_preds[name], test_preds[name] = np.clip(o, 0, 5), np.clip(tr_p, 0, 5), np.clip(te_p, 0, 5)
    rows.append((name, rmse(y, oof[name]), pearsonr(y, oof[name])[0]))
    print(f"{name:16s} CV RMSE = {rows[-1][1]:.4f}   Pearson = {rows[-1][2]:.4f}   zero-detector AUC = {roc_auc_score(zero, p_oof):.3f}")
    return p_oof

p_zero_audio = hurdle("hurdle_audio", np.hstack([X_audio, X_hand]), X_audio)
p_zero_wavlm = hurdle("hurdle_wavlm", np.hstack([X_wavlm, X_hand]), X_wavlm)
HURDLE = ["hurdle_audio", "hurdle_wavlm"]

## 8. Stacking
Three candidate blends are compared by cross-validated RMSE, and the best one is used.

In [ ]:
y_strat = pd.qcut(y, 5, labels=False, duplicates="drop")

def stack_cv_pred(names):
    O = np.column_stack([oof[n] for n in names])
    cv = np.zeros(len(y))
    for tr_idx, va_idx in StratifiedKFold(5, shuffle=True, random_state=7).split(O, y_strat):
        cv[va_idx] = Ridge(alpha=1.0, positive=True).fit(O[tr_idx], y[tr_idx]).predict(O[va_idx])
    return np.clip(cv, 0, 5)

candidates = {"original models": BASE,
              "+ WavLM / tuned models": BASE + NEW,
              "+ hurdle models": BASE + NEW + HURDLE}
cand_cv = {k: stack_cv_pred(v) for k, v in candidates.items()}
for k, v in cand_cv.items():
    print(f"{k:26s} stacked CV RMSE = {rmse(y, v):.4f}   Pearson = {pearsonr(y, v)[0]:.4f}")

best_key = min(cand_cv, key=lambda k: rmse(y, cand_cv[k]))
names = candidates[best_key]
stack_cv = cand_cv[best_key]
print("\nUsing blend:", best_key)

OOF = np.column_stack([oof[n] for n in names])
TRAIN = np.column_stack([train_preds[n] for n in names])
TEST = np.column_stack([test_preds[n] for n in names])
blender = Ridge(alpha=1.0, positive=True).fit(OOF, y)
print("Blend weights (non-zero):")
print(pd.Series(blender.coef_, index=names).round(3)[lambda s: s > 0].to_string())

train_final = np.clip(blender.predict(TRAIN), 0, 5)
test_final = np.clip(blender.predict(TEST), 0, 5)

## 9. Results

In [ ]:
results = pd.DataFrame(rows, columns=["model", "CV_RMSE", "CV_Pearson"])
results.loc[len(results)] = [f"STACKED ({best_key})", rmse(y, stack_cv), pearsonr(y, stack_cv)[0]]
display(results.sort_values("CV_RMSE").round(4).reset_index(drop=True).head(12))

BASELINE_RMSE = float(np.sqrt(np.mean((y - y.mean()) ** 2)))
print("=" * 55)
print("TRAINING RMSE (final model) :", round(rmse(y, train_final), 4))
print("Training Pearson            :", round(pearsonr(y, train_final)[0], 4))
print("5-fold CV RMSE (unseen data):", round(rmse(y, stack_cv), 4))
print("5-fold CV Pearson           :", round(pearsonr(y, stack_cv)[0], 4))
print("Baseline RMSE (mean guess)  :", round(BASELINE_RMSE, 4))
print("=" * 55)

fig, ax = plt.subplots(1, 3, figsize=(17, 4.5))
ax[0].scatter(y, stack_cv, alpha=0.4)
ax[0].plot([0, 5], [0, 5], "r--")
ax[0].set(xlabel="True score", ylabel="Predicted (CV)", title="Predicted vs true")
sns.histplot(y - stack_cv, kde=True, ax=ax[1]); ax[1].set_title("Errors (true - predicted)")
r = results.sort_values("CV_RMSE", ascending=False).tail(12)
ax[2].barh(r["model"], r["CV_RMSE"], color=["tab:red" if "STACK" in m else "tab:blue" for m in r["model"]])
ax[2].set_title("CV RMSE, best 12 models (lower is better)")
plt.tight_layout(); plt.show()

# error split: zero files vs the rest
err2 = (y - stack_cv) ** 2
print(f"Share of squared error from the {int((y == 0).sum())} score-0 files: {err2[y == 0].sum() / err2.sum():.1%}")
print(f"RMSE on score-0 files: {np.sqrt(err2[y == 0].mean()):.3f} | RMSE on the rest: {np.sqrt(err2[y > 0].mean()):.3f}")

In [ ]:
# Evaluation summary generated from the actual results (no hand-typed numbers)
display(Markdown(f'''
## Evaluation summary

| Metric | Value |
|---|---|
| **Training RMSE (final model)** | **{rmse(y, train_final):.4f}** |
| Training Pearson | {pearsonr(y, train_final)[0]:.4f} |
| 5-fold CV RMSE (unseen-data estimate) | {rmse(y, stack_cv):.4f} |
| 5-fold CV Pearson | {pearsonr(y, stack_cv)[0]:.4f} |
| Baseline RMSE (predict the mean) | {BASELINE_RMSE:.4f} |
| Blend used | {best_key} |

Training RMSE is lower than CV RMSE because the final models have seen the training files; the CV RMSE is the honest estimate of unseen-data performance.
'''))

## 10. Predictions and submission file

In [ ]:
submission = test_df[[FILE_COL]].copy()
submission[LABEL_COL] = test_final
submission.to_csv("submission.csv", index=False)

# sanity checks
s = pd.read_csv("submission.csv")
print("shape:", s.shape, "| min:", round(s[LABEL_COL].min(), 3), "| max:", round(s[LABEL_COL].max(), 3),
      "| missing:", int(s[LABEL_COL].isna().sum()), "| duplicate ids:", int(s[FILE_COL].duplicated().sum()))
assert s.shape[0] == len(test_df) and s[LABEL_COL].between(0, 5).all() and not s[LABEL_COL].isna().any()
display(s.head())

## Conclusion
- The final model combines Whisper and WavLM audio embeddings, RoBERTa transcript embeddings and GPT-2 grammar statistics, with a hurdle model for score-0 recordings, stacked into one score in [0, 5]. The exact numbers are in the Evaluation summary above.
- Train and test audio are matched inside their own folders, because some filenames exist in both; matching by name only reads the wrong recording for those files and badly hurts test performance.
- **Limitations:** Whisper tends to clean up grammar mistakes when transcribing, which hides some errors; the acoustic embeddings partly compensate. The 0.5-step human labels also put a floor under achievable RMSE.
- **Possible improvements:** fine-tuning a speech model end to end, attention pooling over frame-level features, a larger Whisper model, and repeated CV seeds.